# Import des différents modules

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import missingno as msno
import re
import seaborn as sns
import matplotlib.pyplot as plt

import bentoml

In [2]:
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV, 
    cross_validate,
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error 
from sklearn.inspection import permutation_importance

#Preprocess
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler,  RobustScaler, MinMaxScaler

#Modèles
from sklearn.dummy import DummyRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR

from sklearn.ensemble import AdaBoostRegressor, BaggingRegressor, GradientBoostingRegressor, RandomForestRegressor

from sklearn.linear_model import Ridge, Lasso, LinearRegression, ElasticNet

from sklearn.tree import DecisionTreeRegressor

# ajout
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold ,cross_validate
from sklearn.pipeline import Pipeline



### Lecture du fichier analysé

In [3]:
df_analyse_ml = pd.read_csv('./data/projet6_analyse.csv')


Lecture du fichier estimation

In [4]:
df_esti = pd.read_csv('./data/projet6_estimation.csv')

In [5]:
df_analyse_ml.info()

<class 'pandas.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   DataYear                1440 non-null   int64  
 1   BuildingType            1440 non-null   str    
 2   PrimaryPropertyType     1440 non-null   str    
 3   Latitude                1440 non-null   float64
 4   Longitude               1440 non-null   float64
 5   NumberofBuildings       1440 non-null   float64
 6   PropertyGFATotal        1440 non-null   int64  
 7   PropertyGFAParking      1440 non-null   int64  
 8   LargestPropertyUseType  1440 non-null   str    
 9   SiteEnergyUse(kBtu)     1440 non-null   float64
 10  TotalGHGEmissions       1440 non-null   float64
 11  BuildingAge             1440 non-null   int64  
 12  mean_GFA_per_floor      1440 non-null   float64
 13  Number_of_Use_Types     1440 non-null   int64  
dtypes: float64(6), int64(5), str(3)
memory usage: 157.6

In [6]:
df_esti.info()

<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   DataYear                96 non-null     int64  
 1   BuildingType            96 non-null     str    
 2   PrimaryPropertyType     96 non-null     str    
 3   Latitude                96 non-null     float64
 4   Longitude               96 non-null     float64
 5   NumberofBuildings       96 non-null     float64
 6   PropertyGFATotal        96 non-null     int64  
 7   PropertyGFAParking      96 non-null     int64  
 8   LargestPropertyUseType  96 non-null     str    
 9   SiteEnergyUse(kBtu)     96 non-null     float64
 10  TotalGHGEmissions       96 non-null     float64
 11  BuildingAge             96 non-null     int64  
 12  mean_GFA_per_floor      96 non-null     float64
 13  Number_of_Use_Types     96 non-null     int64  
dtypes: float64(6), int64(5), str(3)
memory usage: 10.6 KB


In [7]:
print (df_analyse_ml.isna().sum() )

DataYear                  0
BuildingType              0
PrimaryPropertyType       0
Latitude                  0
Longitude                 0
NumberofBuildings         0
PropertyGFATotal          0
PropertyGFAParking        0
LargestPropertyUseType    0
SiteEnergyUse(kBtu)       0
TotalGHGEmissions         0
BuildingAge               0
mean_GFA_per_floor        0
Number_of_Use_Types       0
dtype: int64


## Comparaison des méthodes



**RMSE** (Root Mean Squared Error) : Mesure la taille moyenne des écarts entre valeurs prédites et réelles (dans l'unité de la variable).Objectif : À minimiser (plus il est proche de 0, plus le modèle est exact).

**MSE** (Mean Squared Error) : Moyenne des carrés des erreurs, correspondant à la variance résiduelle. Sert de base mathématique à minimiser lors de l'entraînement d'une régression. Penalise plus fortement les grands écarts.Objectif : À minimiser.

**MAE** (Mean Absolute Error) : Moyenne des valeurs absolues des écarts. Donnes une mesure directe de l'erreur moyenne sans sur-pénaliser les valeurs aberrantes.Objectif : À minimiser.

**R2** (Coefficient de détermination) : Mesure la qualité de la corrélation entre les prédictions et la réalité (exprime la proportion de variance expliquée).Objectif : À maximiser (plus il est proche de 1, plus les prédictions sont fidèles aux données réelles).


In [8]:
df_test1 = df_analyse_ml.copy()
# ==============================================================================
# 1. PRÉPARATION DES DONNÉES (X et y)
# ==============================================================================
# La cible y avec transformation log (lissage de l'asymétrie)
y = np.log1p(df_test1[["TotalGHGEmissions", "SiteEnergyUse(kBtu)"]])


# Les features X (tout sauf la cible)
X = df_test1.drop(columns=["TotalGHGEmissions","SiteEnergyUse(kBtu)"])

num_features = X.select_dtypes( include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(
    include=["category","str","object"]
).columns.tolist()


# ==============================================================================
# 2. PRÉPARATEUR DE DONNÉES (ColumnTransformer)
# ==============================================================================
preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), num_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            cat_features,
        ),
    ]
)


# ==============================================================================
# 3. DÉFINITION DE LA VALIDATION CROISÉE (K-Fold)
# ==============================================================================
# 5 plis (folds), avec mélange aléatoire pour éviter tout biais d'ordre
kf = KFold(n_splits=5, shuffle=True, random_state=42)


# ==============================================================================
# 4. DICTIONNAIRE DES ALGORITHMES À TESTER
# ==============================================================================

models = {
    "Régression Linéaire": LinearRegression(),
    "Régression Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": MultiOutputRegressor (GradientBoostingRegressor(n_estimators=100, random_state=42)),
}


# ==============================================================================
# 5. ÉVALUATION PAR VALIDATION CROISÉE POUR CHAQUE ALGORITHME
# ==============================================================================
results = []

for name, model in models.items():
    # Création du Pipeline complet (Préprocessing + Modèle)
    full_pipeline = Pipeline(
        steps=[("preprocessor", preprocessor), ("model", model)]
    )

    # découpage
    # Cross-validation évaluant le R² et la RMSE (Root Mean Squared Error)
    cv_results = cross_validate(
        full_pipeline,
        X,
        y,
        cv=kf,
        scoring={
            "r2": "r2",
            "rmse": "neg_root_mean_squared_error",
            "mae": "neg_mean_absolute_error"
        },
        return_train_score=False,
    )

    # Récupération des moyennes des métriques
    r2_mean = cv_results["test_r2"].mean()
    r2_std = cv_results["test_r2"].std()
    rmse_mean = -cv_results["test_rmse"].mean()
    mae_mean = -cv_results["test_mae"].mean()

    results.append(
        {
            "Algorithme": name,
            "R² (Moyenne)": round(r2_mean, 4),
            "R² (Écart-type)": round(r2_std, 4),
            "RMSE (Moyenne)": round(rmse_mean, 4),
            "MAE (Moyenne)": round(mae_mean, 4)
        }
    )

# Affichage du tableau récapitulatif
df_results = pd.DataFrame(results)
display(df_results)

,Algorithme,R² (Moyenne),R² (Écart-type),RMSE (Moyenne),MAE (Moyenne)
0,Régression Linéaire,0.3892,0.0932,0.9281,0.7007
1,Régression Ridge,0.4000,0.0838,0.9203,0.6957
2,Random Forest,0.5268,0.0214,0.8151,0.6292
3,Gradient Boosting,0.5326,0.0139,0.8098,0.6301


In [9]:
df_test2 = df_analyse_ml.copy()
# PRÉPARATION DES DONNÉES (X et y)
# La cible y avec transformation log (lissage de l'asymétrie)
y = np.log1p(df_test2[["TotalGHGEmissions", "SiteEnergyUse(kBtu)"]])
#y = df_test2["TotalGHGEmissions"]
# Les features X (tout sauf la cible)
X = df_test2.drop(columns=["TotalGHGEmissions","SiteEnergyUse(kBtu)"])

num_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()
cat_features = X.select_dtypes(
    include=["category","str","object"]
).columns.tolist()


# PRÉPARATEUR DE DONNÉES (ColumnTransformer)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), num_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            cat_features,
        ),
    ]
)



#  DÉFINITION DE LA VALIDATION CROISÉE (K-Fold)

# 5 plis (folds), avec mélange aléatoire pour éviter tout biais d'ordre
kf = KFold(n_splits=5, shuffle=True, random_state=42)


# DICTIONNAIRE DES ALGORITHMES À TESTER

models = {
    'dummy_reg': DummyRegressor(),
    'Régression Linéaire': LinearRegression(),
    'Régression Ridge' : Ridge(alpha=1.0),
    'lasso' : MultiOutputRegressor (Lasso(random_state=42)),
    'dec_tree':  DecisionTreeRegressor(random_state=42),
    'svr' : MultiOutputRegressor (SVR()),
    'adaboost' : MultiOutputRegressor (AdaBoostRegressor(random_state=42)),
    'bagging': BaggingRegressor(random_state=42),
    'Gradient Boosting':  MultiOutputRegressor (GradientBoostingRegressor(n_estimators=100, random_state=42)),
    'Random Forest' : RandomForestRegressor(n_estimators=100, random_state=42),
    'Knregressor' : KNeighborsRegressor()
}

#  ÉVALUATION PAR VALIDATION CROISÉE POUR CHAQUE ALGORITHME

results = []

for name, model in models.items():
    # Création du Pipeline complet (Préprocessing + Modèle)
    full_pipeline = Pipeline(
        steps=[("preprocessor", preprocessor), ("model", model)]
    )

    # découpage
    # Cross-validation évaluant le R² et la RMSE (Root Mean Squared Error)
    cv_results = cross_validate(
        full_pipeline,
        X,
        y,
        cv=kf,
        scoring={
            "r2": "r2",
            "rmse": "neg_root_mean_squared_error",
            "mae": "neg_mean_absolute_error"
        },
        return_train_score=False,
    )

    # Récupération des moyennes des métriques
    r2_mean = cv_results["test_r2"].mean()
    r2_std = cv_results["test_r2"].std()
    rmse_mean = -cv_results["test_rmse"].mean()
    mae_mean = -cv_results["test_mae"].mean()

    results.append(
        {
            "Algorithme": name,
            "R² (Moyenne)": round(r2_mean, 4),
            "R² (Écart-type)": round(r2_std, 4),
            "RMSE (Moyenne)": round(rmse_mean, 4),
            "MAE (Moyenne)": round(mae_mean, 4)
        }
    )

# Affichage du tableau récapitulatif
df_results = pd.DataFrame(results)
display(df_results)

,Algorithme,R² (Moyenne),R² (Écart-type),RMSE (Moyenne),MAE (Moyenne)
0,dummy_reg,-0.0096,0.0075,1.1946,0.9675
1,Régression Linéaire,0.3892,0.0932,0.9281,0.7007
2,Régression Ridge,0.4000,0.0838,0.9203,0.6957
3,lasso,0.0877,0.0127,1.1362,0.9081
4,dec_tree,0.1560,0.0964,1.0837,0.8373
5,svr,0.1030,0.0168,1.1271,0.8967
6,adaboost,0.4247,0.0173,0.8993,0.7204
7,bagging,0.4942,0.0188,0.8427,0.6502
8,Gradient Boosting,0.5326,0.0139,0.8098,0.6301
9,Random Forest,0.5268,0.0214,0.8151,0.6292


In [10]:
df_test3 = df_analyse_ml.copy()

# 1. PRÉPARATION DE X ET y (2 cibles)
y = np.log1p(df_test3[["TotalGHGEmissions", "SiteEnergyUse(kBtu)"]])
X = df_test3.drop(columns=["TotalGHGEmissions", "SiteEnergyUse(kBtu)"])

# 2. DÉFINITION DE LA VALIDATION CROISÉE (K-Fold)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 3. SÉPARATION EN TRAIN (80%) ET TEST (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Identification automatique des colonnes
num_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()
cat_features = X_train.select_dtypes(
    exclude=["int64", "float64"]
).columns.tolist()

# Préparateur
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            cat_features,
        ),
    ]
)

# 4. PIPELINE AVEC MULTIOUTPUTREGRESSOR
full_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            MultiOutputRegressor(
                GradientBoostingRegressor(random_state=42)
            ),
        ),
    ]
)

# 5. GRID SEARCH (Attention au préfixe model__estimator__)
param_grid = {
    "model__estimator__n_estimators": [100, 200, 300],
    "model__estimator__learning_rate": [0.03, 0.05, 0.1],
    "model__estimator__max_depth": [3, 4, 5],
    "model__estimator__subsample": [0.8, 1.0],
}

grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=kf,
    scoring="r2",
    n_jobs=-1,
    verbose=1,
)

print("--- Entraînement et optimisation sur le jeu Train (80%) ---")
grid_search.fit(X_train, y_train)

print(f"\n Meilleur score R² moyen en CV : {grid_search.best_score_:.4f}")
print(f" Meilleurs hyperparamètres : {grid_search.best_params_}")


# 6. ÉVALUATION ET DÉCOUPLAGE DES MÉTRIQUES SUR LE TEST (20%)
meilleur_model = grid_search.best_estimator_
y_pred_test = meilleur_model.predict(X_test)

# Calcul des métriques par cible avec multioutput='raw_values'
r2_final = r2_score(y_test, y_pred_test, multioutput="raw_values")
rmse_final = np.sqrt(
    mean_squared_error(y_test, y_pred_test, multioutput="raw_values")
)
mae_final = mean_absolute_error(y_test, y_pred_test, multioutput="raw_values")

print("\n" + "=" * 60)
print("RÉSULTATS DE VALIDATION FINALE SUR LE JEU DE TEST (20%)")
print("=" * 60)

print("\n--- ÉMISSIONS DE CO2 (TotalGHGEmissions) ---")
print(f"R² (Test final)   : {r2_final[0]:.4f}")
print(f"RMSE (log)        : {rmse_final[0]:.4f}")
print(f"MAE (log)         : {mae_final[0]:.4f} (~{np.expm1(mae_final[0]):.2f} tCO2eq)")

print("\n--- CONSOMMATION D'ÉNERGIE (SiteEnergyUse(kBtu)) ---")
print(f"R² (Test final)   : {r2_final[1]:.4f}")
print(f"RMSE (log)        : {rmse_final[1]:.4f}")
print(f"MAE (log)         : {mae_final[1]:.4f} (~{np.expm1(mae_final[1]):.2f} kBtu)")

--- Entraînement et optimisation sur le jeu Train (80%) ---
Fitting 5 folds for each of 54 candidates, totalling 270 fits

 Meilleur score R² moyen en CV : 0.5455
 Meilleurs hyperparamètres : {'model__estimator__learning_rate': 0.03, 'model__estimator__max_depth': 3, 'model__estimator__n_estimators': 300, 'model__estimator__subsample': 0.8}

RÉSULTATS DE VALIDATION FINALE SUR LE JEU DE TEST (20%)

--- ÉMISSIONS DE CO2 (TotalGHGEmissions) ---
R² (Test final)   : 0.3982
RMSE (log)        : 1.0426
MAE (log)         : 0.8280 (~1.29 tCO2eq)

--- CONSOMMATION D'ÉNERGIE (SiteEnergyUse(kBtu)) ---
R² (Test final)   : 0.6541
RMSE (log)        : 0.6781
MAE (log)         : 0.5145 (~0.67 kBtu)


### Résultat sur le data frame estimation 

In [11]:
df_test3 = df_esti .copy()

# PRÉPARATION DE DF_ESTI

# La valeur estimée réelle (passée en log1p comme pendant l'entraînement)
y_estime = np.log1p(df_test3[["TotalGHGEmissions", "SiteEnergyUse(kBtu)"]])

# Les variables explicatives de df_esti
X_esti = df_test3.drop(columns=["TotalGHGEmissions","SiteEnergyUse(kBtu)"])


#  PRÉDICTION  MEILLEUR MODÈLE

y_pred_modele = meilleur_model.predict(X_esti)


# CALCUL DES MÉTRIQUES D'ALIGNEMENT

r2_accord = r2_score(y_estime, y_pred_modele , multioutput ="raw_values")
mae_accord = mean_absolute_error(y_estime, y_pred_modele , multioutput ="raw_values")

mae_co2 = mae_accord[0]
mae_energie = mae_accord[1]
# Conversion de l'erreur brute (sur l'échelle réelle )
                              
erreur_co2 = np.expm1(mae_co2)
erreur_energie = np.expm1(mae_energie)

print("=" * 60)
print("ÉVALUATION DE LA QUALITÉ DES ESTIMATIONS DU DF_ESTI")
print("=" * 60)
print ("--- Emission CO2 ----")
print (f"R²  : {r2_accord[0]:.4f}" )                          
print (f"MAE (log)  : {mae_co2:.4f} (~{erreur_co2:.2f} tCO2e)" )
print("\n")
print ("--- consommation énergétique ----")
print(f"R²  : {r2_accord[1]:.4f}")                          
print(f"MAE (log)  : {mae_energie:.4f} (~{erreur_energie:.2f} kBtu)" )


ÉVALUATION DE LA QUALITÉ DES ESTIMATIONS DU DF_ESTI
--- Emission CO2 ----
R²  : 0.2717
MAE (log)  : 0.6827 (~0.98 tCO2e)


--- consommation énergétique ----
R²  : -0.1931
MAE (log)  : 2.1202 (~7.33 kBtu)


### Sauvegarde du model avec bentoml

In [12]:
# Retirer le commentaire pour sauvegarder le meilleur model après les différents tests 

In [13]:
save_bentoml = bentoml.sklearn.save_model(name="seattle_co2" , model = meilleur_model )